# Glue Interactive Sessions - Processamento Streaming (CDC)

Este notebook conecta em uma **sessão interativa do AWS Glue** (conta `331504768406`, região `us-east-1`) e executa os **mesmos passos do modo streaming** da pipeline (`main._run_streaming`): leitura via `reader.read(mode='streaming')` (Spark Structured Streaming), processamento de cada micro-batch com validação de qualidade, escrita de rejeitados e MERGE Delta, usando `writeStream.foreachBatch`.

**Pré-requisitos:**

- Kernel **Glue PySpark** local: `pip install jupyter boto3 aws-glue-sessions` e depois `install-glue-kernels`. Abra com `jupyter notebook` e escolha o kernel `Glue PySpark`.
- Credenciais AWS válidas com acesso ao Glue e S3 (usuário `lake-admin` / grupo `datalake-admins`).
- Role `role-datalake-analytics` com as permissões de interactive sessions (já provisionadas via Terraform em `infra/iam.tf`).
- `helpers.zip` publicado no bucket workspace (`dependencies/helpers.zip`).

> **Delta Lake**: a célula `%%configure` abaixo é obrigatória - o `writer.py` do projeto importa `delta.tables`. Sem ela a importação falha com `ModuleNotFoundError: No module named 'delta'`.
>
> Ajuste o ARN da role e o bucket abaixo se a conta AWS for diferente de `331504768406`.
>
> O notebook usa **checkpoint de teste** (sufixo `interactive-test/`) para não conflitar com o job de produção `glue-flight-radar-stream-cdc`. Prefira executá-lo com o job de streaming parado.

In [ ]:
# 1) Configuração da sessão (magics do kernel AWS Glue)
# A célula seguinte (%%configure) habilita o Delta Lake. Cell magics
# (%%configure/%%tags) devem ser a primeira linha da célula, sem comentários.
%glue_version 5.0
%iam_role arn:aws:iam::331504768406:role/role-datalake-analytics
%region us-east-1
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%session_id_prefix flight-radar-stream

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore"
}

In [ ]:
%%tags
{"Environment": "production", "Project": "flight-radar-glue", "Mode": "streaming"}

In [ ]:
# Dependências do projeto (helpers.zip já publicado no workspace)
%extra_py_files s3://lakehouse-workspace-331504768406/dependencies/helpers.zip

In [ ]:
# Instanciar a SparkSession / GlueContext
# Em sessões interativas o Spark já existe; getOrCreate retorna a mesma sessão.
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

print('Spark version:', spark.version)

In [ ]:
# Carregar a configuração e escolher a tabela a testar
import boto3
from processor import Processor
from config_models import Config

account_id = boto3.client('sts').get_caller_identity()['Account']
config = Config.from_s3(f's3://lakehouse-workspace-{account_id}/config/config.json')

processor = Processor(spark)
source = config.get_source('aircraft')
target = source.target
print(source.source, '->', f'{target.database}.{target.table}')
print('CDC path:', source.cdc_source_location)

## Passos equivalentes ao `main._run_streaming`

O `main._run_streaming` inicia uma query Spark Structured Streaming por tabela: lê o CDC via `reader.read(mode='streaming')` e usa `foreachBatch` para validar, escrever rejeitados e fazer o MERGE Delta em cada micro-batch. Aqui executamos a mesma sequência em células separadas.

Para gerar micro-batches durante o teste, **solte arquivos Parquet CDC** (com colunas `Op` e `dms_timestamp`) no `CDC path` acima.

In [ ]:
# Passo 1 - Leitura streaming do CDC (reader.read com mode='streaming')
# Lê de source.cdc_source_location com includeExistingFiles=false e
# cleanSource=archive (arquivos processados vão para *_archive/).
stream_df = processor._reader.read(source, mode='streaming')
print('isStreaming:', stream_df.isStreaming)

In [ ]:
# Passo 2 - Função process_batch (validação + rejects + MERGE Delta)
# Espelha a closure process_batch do main._run_streaming.
def process_batch(df, batch_id, src=source, tgt=target):
    print(f'[{src.source}] batch_id={batch_id} rows={df.count()}')
    valid_df, rejects_df = processor._data_quality.validate(df, tgt, src)
    processor._writer.write_rejects(rejects_df, tgt)
    processor._writer.write(valid_df, tgt, src)
    print(f'[{src.source}] batch {batch_id} done')

In [ ]:
# Passo 3 - Iniciar a query streaming (foreachBatch + trigger 30s)
# Usa checkpoint de teste para não conflitar com o job de produção.
test_checkpoint = source.checkpoint_location.rstrip('/') + '/interactive-test/'

query = (
    stream_df.writeStream
    .foreachBatch(process_batch)
    .outputMode('append')
    .trigger(processingTime='30 seconds')
    .option('checkpointLocation', test_checkpoint)
    .start()
)
print('Query iniciada - id:', query.id)
print('Checkpoint:', test_checkpoint)

In [ ]:
# Passo 4 - Observar os micro-batches por alguns segundos e parar a query
# Solte arquivos CDC no caminho acima durante este intervalo para ver os
# prints do process_batch. A query é parada ao final da célula.
import time

print('Processando micro-batches por 60 segundos...')
time.sleep(60)
print('Status:', query.status)
query.stop()
print('Query parada.')

In [ ]:
# Verificação - ler a tabela Delta final (via Glue Catalog)
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, f'{target.database}.{target.table}')
print('Linhas na tabela:', dt.toDF().count())
dt.toDF().show(5)

In [ ]:
# Status da sessão (mostra tags, role, workers, região)
%status

In [ ]:
# Encerrar a sessão quando terminar
%stop_session